# DATALAB MEF Maroc - UC-S1 - Etape 2
## Caracteristiques, selection, VAR/VECM et validation

Notebook maitre : orchestre le protocole temporel, la construction des
caracteristiques, les tests de stationnarite/cointegration, et affiche la
synthese du portefeuille predictif (baselines, Elastic Net, panel pooled)
evalue selon le protocole du cadrage (Ch.6.5.3).

Le calcul complet (grille 7 cibles x 5 horizons x 3 modeles) est effectue
par les scripts `run_etape2_part1_audit_var.py`, `run_etape2_part2_models.py`
et `run_etape2_part3_synthese.py` (executables independamment depuis un
terminal). Ce notebook reexecute les etapes rapides (donnees, stationnarite,
VAR/VECM) et charge/affiche les resultats de la grille complete de
modelisation pour la lecture et le commentaire.

Convention de statut : **implemente**, **execute**, **evalue**, **valide**,
**non concluant**, **bloque**.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import warnings
warnings.filterwarnings("ignore", message="KMeans is known to have a memory leak on Windows with MKL.*", category=UserWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

import json
import pandas as pd
from IPython.display import display, Image

from src import io_utils
from src.etape2 import cointegration_var as cv
from src.etape2 import data_extended as dext
from src.etape2 import features as feat
from src.etape2 import stationarity as stn

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)


C:\Users\MR KITOHOU\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Configuration et jeu de travail etendu (30 indicateurs)

In [2]:
cfg1 = io_utils.load_config("configs/config.yaml")
cfg2 = io_utils.load_config("configs/config_etape2.yaml")
logger = io_utils.setup_logging({"logging": cfg2["logging"] | {"log_file": cfg2["paths"]["log_file"]}})

extended_df = dext.load_extended_work_base(cfg1, cfg2, logger)
ok = dext.verify_against_etape1_work_base(extended_df, cfg1, logger)
print("Jeu etendu :", extended_df.shape, "| coherent avec l'etape 1 :", ok)
extended_df.head()


2026-09-14 01:44:43,939 [INFO] Base source chargee : C:\Users\MR KITOHOU\Desktop\Work_DAVID_DG_Mohamed\Data_collection\data\base_B_revised_wide.csv (shape=(405, 32), sha256=083587b9b54e0b92)


2026-09-14 01:44:43,941 [INFO] Jeu de travail etendu construit : shape=(306, 32) (28 indicateurs etape 1 + DETTE_PUBLIQUE + TCER)


2026-09-14 01:44:43,945 [INFO] Fichier de travail charge : C:\Users\MR KITOHOU\Desktop\Work_DAVID_DG_Mohamed\Data_collection\base_MEF_1991_2024_9pays_28indicateurs.csv (shape=(306, 30), sha256=af4e19eec6159577)


2026-09-14 01:44:43,960 [INFO] Verification jeu etendu vs fichier de travail etape 1 : OK (28 colonnes comparees)


Jeu etendu : (306, 32) | coherent avec l'etape 1 : True


,country_iso3,year,BALANCE_COURANTE,CRDE,CROISSANCE_PIB,DETO,DETTE_PUBLIQUE,EMAG,EMIN,EMSE,EVNA,EXPO,EXPORT_MARCHANDISES_USD,FBCF,IDEE,IMPO,INFLATION_CPI,LFPRF,OUVERTURE_COM,POP_ACTIVE_TOTALE,RETO,SOBG,SOPR,TAAC,TACH,TAUX_EMPLOI_15P,TCER,UNFE,VAAG,VAIN,VASE,WSWF
0,CHL,1991,-0.255543,1.662900,7.756202,21.248734,37.1,16.722591,26.760737,56.516652,74.082,30.984879,8.942000e+09,20.749222,2.151844,26.522668,21.784412,31.873,57.507547,5120213.0,22.789961,1.541227,3.785388,56.670,5.23,50.416,88.089640,5.700,9.162728,36.995475,46.137989,75.084478
1,CHL,1992,-2.056130,1.612446,11.501931,20.932553,30.5,15.831663,26.649239,57.519078,74.367,28.595070,1.000700e+10,23.498437,2.011436,26.946677,15.425806,33.205,55.541748,5322228.0,23.040160,2.107607,3.884624,57.931,4.35,51.887,92.925148,5.340,9.089227,34.817031,47.591766,75.034777
2,CHL,1993,-5.111050,1.555755,6.685584,21.217570,28.1,14.487220,27.288794,58.223986,74.679,25.545056,9.199000e+09,26.218885,2.071295,27.368798,12.727767,34.088,52.913853,5480115.0,22.633006,1.415435,2.957829,58.680,4.49,52.349,94.663585,5.115,7.929480,34.541908,48.691699,74.985294
3,CHL,1994,-2.762966,1.505947,5.076804,20.621123,22.6,14.157554,26.223050,59.619397,74.973,27.253018,1.160400e+10,24.720992,4.495435,25.554666,11.443120,34.784,52.807684,5623118.0,22.058509,1.437386,2.797999,59.244,5.87,51.965,98.070098,6.790,7.444784,36.305786,48.904877,74.988510
4,CHL,1995,-1.828220,1.463954,9.255428,19.451047,17.3,13.704047,26.138134,60.157801,75.298,28.480603,1.602400e+10,25.619462,4.005666,26.206880,8.232630,35.896,54.687484,5812124.0,22.551988,3.100941,4.058524,60.260,4.70,53.397,103.953008,5.246,6.967403,37.849982,48.090406,75.245868


**Statut : execute.** Le jeu de travail etendu reintegre `DETTE_PUBLIQUE`
et `TCER` depuis la base source (`base_B_revised_wide.csv`), verifies
identiques aux 28 colonnes du fichier de travail de l'etape 1 sur la
fenetre 1991-2024/9 pays. Le Maroc a une couverture complete (0 manquant)
sur les deux variables reintegrees. Decision documentee dans
`docs/SUIVI_MODELISATION.md` (met a jour le point ouvert #1 de l'etape 1).

## 2. Stationnarite (Maroc uniquement, ADF + KPSS)

In [3]:
mar = extended_df[extended_df["country_iso3"] == "MAR"].set_index("year").sort_index()
all_vars = sorted(set(list(cfg2["targets"].keys()) + ["SOPR", "DETO", "RETO", "TCER", "IDEE", "TAAC", "POP_ACTIVE_TOTALE", "EMAG", "EMIN"]))
stationarity_table = stn.classify_all(mar, all_vars)
display(stationarity_table[["variable", "n_obs", "adf_pvalue_niveau", "kpss_pvalue_niveau", "ordre_integration", "confiance"]])


,variable,n_obs,adf_pvalue_niveau,kpss_pvalue_niveau,ordre_integration,confiance
0,BALANCE_COURANTE,34,3.419666e-01,0.100000,indetermine,"contradictoire (ADF non-stationnaire, KPSS sta..."
1,CROISSANCE_PIB,34,8.426353e-20,0.100000,I(0),concordant
2,DETO,34,7.504850e-01,0.017227,I(1),concordant
3,DETTE_PUBLIQUE,34,2.042345e-01,0.100000,indetermine,"contradictoire (ADF non-stationnaire, KPSS sta..."
4,EMAG,34,9.987580e-01,0.010000,indetermine,"schema non standard, a examiner manuellement"
5,EMIN,34,8.389330e-01,0.020634,I(1),concordant
6,FBCF,34,4.680810e-01,0.047472,I(1),concordant
7,IDEE,34,7.464026e-02,0.100000,indetermine,"contradictoire (ADF non-stationnaire, KPSS sta..."
8,INFLATION_CPI,34,4.813633e-02,0.100000,I(0),concordant
9,POP_ACTIVE_TOTALE,34,4.282030e-03,0.010000,indetermine,"contradictoire (ADF stationnaire, KPSS non-sta..."


**Lecture.** Deux cibles sont clairement I(0) (CROISSANCE_PIB,
INFLATION_CPI), six variables I(1) (dont FBCF, TACH). Cinq variables restent
**indeterminees** (ADF et KPSS ne rejettent aucun des deux H0 -- puissance
statistique probablement faible avec N=34), dont BALANCE_COURANTE,
DETTE_PUBLIQUE et TCER. Trois variables (SOBG, SOPR, EMAG) presentent un
schema non standard. **Aucune serie n'est forcee** vers un ordre
d'integration par defaut : ce constat conditionne directement la decision
VAR/VECM ci-dessous.

## 3. Cointegration et admissibilite VAR/VECM

In [4]:
integration_map = dict(zip(stationarity_table["variable"], stationarity_table["ordre_integration"]))

rows = []
for system in cfg2["var_candidate_systems"]:
    variables = system["variables"]
    sys_integration = {v: integration_map.get(v, "inconnu") for v in variables}
    data = mar[variables].dropna()
    decision = cv.decide_system_type(sys_integration, None, len(variables))
    rows.append({"systeme": system["name"], "variables": ", ".join(variables), "ordres": sys_integration, "T": len(data), "decision": decision["type"], "motif": decision["motif"]})
pd.DataFrame(rows)


,systeme,variables,ordres,T,decision,motif
0,dette_solde,"DETTE_PUBLIQUE, SOBG","{'DETTE_PUBLIQUE': 'indetermine', 'SOBG': 'ind...",34,reexamen,Ordres d'integration heterogenes ou indetermin...
1,solde_compte_courant,"SOBG, BALANCE_COURANTE","{'SOBG': 'indetermine', 'BALANCE_COURANTE': 'i...",34,reexamen,Ordres d'integration heterogenes ou indetermin...
2,croissance_investissement,"CROISSANCE_PIB, FBCF","{'CROISSANCE_PIB': 'I(0)', 'FBCF': 'I(1)'}",34,reexamen,Ordres d'integration heterogenes ou indetermin...
3,compte_courant_change,"BALANCE_COURANTE, TCER","{'BALANCE_COURANTE': 'indetermine', 'TCER': 'i...",34,reexamen,Ordres d'integration heterogenes ou indetermin...


**Statut : evalue.** Les quatre systemes candidats (dette~solde,
solde~compte courant, croissance~FBCF, compte courant~TCER) sont **tous
rejetes en amont** : ordres d'integration heterogenes ou indetermines.
Aucun ne remplit la condition prealable ("tous I(1) confirmes") pour
seulement calculer un rang de Johansen. **Aucun VAR ni VECM n'est estime
sur le Maroc dans cette etape** -- decision coherente avec l'audit de
modelisabilite anterieur, qui proscrivait deja VAR/VECM Maroc-seul avec ce
volume de donnees. Detail complet : `docs/ETAPE2_PROTOCOLE_ET_CARACTERISTIQUES.md`
section 6.

## 4. Caracteristiques causales : illustration (CROISSANCE_PIB)

In [5]:
dfc = extended_df[extended_df.country_iso3 == "MAR"]
ff = feat.build_feature_frame(dfc, ["CROISSANCE_PIB", "FBCF"], max_lags=3, ma_windows=[3, 5], year_start=1991, year_end=2024)
ff = feat.drop_all_nan_columns(ff)
ff.loc[2015:2020, [c for c in ff.columns if c.startswith("CROISSANCE_PIB")]]


,CROISSANCE_PIB_lag0,CROISSANCE_PIB_lag1,CROISSANCE_PIB_lag2,CROISSANCE_PIB_lag3,CROISSANCE_PIB_diff1,CROISSANCE_PIB_ma3,CROISSANCE_PIB_std3,CROISSANCE_PIB_ma5,CROISSANCE_PIB_std5
year,,,,,,,,,
2015,4.344583,2.719244,4.122213,3.062344,1.625339,3.301267,0.731364,3.785601,1.104493
2016,0.521186,4.344583,2.719244,4.122213,-3.823397,3.728680,0.881239,3.954606,1.114389
2017,5.057898,0.521186,4.344583,2.719244,4.536713,2.528338,1.918835,2.953914,1.523473
2018,3.065641,5.057898,0.521186,4.344583,-1.992257,3.307889,2.439568,3.353025,1.795997
2019,2.890975,3.065641,5.057898,0.521186,-0.174667,2.881575,2.273950,3.141710,1.744283
2020,-7.178207,2.890975,3.065641,5.057898,-10.069182,3.671505,1.203824,3.176057,1.735554


Verification visuelle de causalite : `CROISSANCE_PIB_lag1` a l'annee O
doit correspondre exactement a `CROISSANCE_PIB_lag0` a l'annee O-1 (aucune
information posterieure a O n'entre dans les caracteristiques a l'origine
O). Teste formellement dans `tests/test_etape2_integrity.py::test_feature_frame_is_causal_no_future_values`.

## 5. Portefeuille predictif : synthese de la grille complete (7 cibles x 5 horizons)

In [6]:
tables_dir = Path(cfg2["paths"]["outputs_tables_dir"])
eval_table = pd.read_csv(tables_dir / "e2_11_evaluation_vs_baseline.csv")
best_table = pd.read_csv(tables_dir / "e2_13_meilleur_modele_par_cible_horizon.csv")
print("Lignes d'evaluation (cible x horizon x modele x bloc) :", len(eval_table))
print("Meilleur modele par (cible, horizon), bloc final :", len(best_table))
best_table[["cible", "horizon", "famille", "modele", "n_previsions", "statut", "verdict"]]


Lignes d'evaluation (cible x horizon x modele x bloc) : 210
Meilleur modele par (cible, horizon), bloc final : 35


,cible,horizon,famille,modele,n_previsions,statut,verdict
0,BALANCE_COURANTE,1,niveau_ratio,elasticnet,4,evalue,"RMSE relative < baseline et pas de derive, DM ..."
1,BALANCE_COURANTE,2,niveau_ratio,elasticnet,4,evalue,"RMSE relative < baseline et pas de derive, DM ..."
2,BALANCE_COURANTE,3,niveau_ratio,panel_avec_effets_fixes,4,evalue,RMSE relative < baseline MAIS derive systemati...
3,BALANCE_COURANTE,4,niveau_ratio,panel_avec_effets_fixes,4,evalue,RMSE relative < baseline MAIS derive systemati...
4,BALANCE_COURANTE,5,niveau_ratio,panel_avec_effets_fixes,4,evalue,RMSE relative < baseline MAIS derive systemati...
5,CROISSANCE_PIB,1,variation,elasticnet,4,evalue,"gain >= seuil observe, DM non significatif ou ..."
6,CROISSANCE_PIB,2,variation,panel_leave_one_country_out,4,evalue,"gain >= seuil observe, DM non significatif ou ..."
7,CROISSANCE_PIB,3,variation,panel_avec_effets_fixes,4,evalue,"gain >= seuil observe, DM non significatif ou ..."
8,CROISSANCE_PIB,4,variation,panel_avec_effets_fixes,4,evalue,"gain >= seuil observe, DM non significatif ou ..."
9,CROISSANCE_PIB,5,variation,panel_leave_one_country_out,4,evalue,"gain >= seuil observe, DM non significatif ou ..."


**Statut : evalue** pour les 35 combinaisons (7 cibles x 5 horizons).
Aucune n'atteint le statut "valide" (revue metier absente, effectif final de
seulement 4 previsions par combinaison). Resume :
- 14/15 combinaisons "variation" (CROISSANCE_PIB, INFLATION_CPI, TACH)
  atteignent le seuil de gain de 10% vs marche aleatoire sur le bloc final
  (seule exception : TACH a h=1).
- 13/20 combinaisons "niveau/ratio" ont un modele avec RMSE relative
  inferieure a la baseline ET sans derive systematique.
- Aucun test de Diebold-Mariano n'est calculable sur le bloc final (n=4
  systematiquement sous le seuil de 8) : tous les gains sont des
  **observations descriptives**, pas des resultats statistiquement
  etablis a ce stade.

In [7]:
display(eval_table[(eval_table.cible == "CROISSANCE_PIB") & (eval_table.bloc == "finale")][
    ["cible", "horizon", "modele", "n_previsions", "rmse_modele", "rmse_baseline", "gain_rmse_vs_baseline_pct", "seuil_gain_atteint", "dm_statut"]
])


,cible,horizon,modele,n_previsions,rmse_modele,rmse_baseline,gain_rmse_vs_baseline_pct,seuil_gain_atteint,dm_statut
1,CROISSANCE_PIB,1,elasticnet,4,2.794958,8.347698,66.518219,True,non_calcule
3,CROISSANCE_PIB,1,panel_avec_effets_fixes,4,3.894616,8.347698,53.345027,True,non_calcule
5,CROISSANCE_PIB,1,panel_leave_one_country_out,4,4.257120,8.347698,49.002469,True,non_calcule
7,CROISSANCE_PIB,2,elasticnet,4,2.424918,5.759957,57.900420,True,non_calcule
9,CROISSANCE_PIB,2,panel_avec_effets_fixes,4,2.235872,5.759957,61.182480,True,non_calcule
11,CROISSANCE_PIB,2,panel_leave_one_country_out,4,2.198295,5.759957,61.834868,True,non_calcule
13,CROISSANCE_PIB,3,elasticnet,4,2.301137,6.392486,64.002480,True,non_calcule
15,CROISSANCE_PIB,3,panel_avec_effets_fixes,4,2.292077,6.392486,64.144201,True,non_calcule
17,CROISSANCE_PIB,3,panel_leave_one_country_out,4,2.317007,6.392486,63.754217,True,non_calcule
19,CROISSANCE_PIB,4,elasticnet,4,2.331005,5.747183,59.440905,True,non_calcule


In [8]:
display(eval_table[(eval_table.cible == "DETTE_PUBLIQUE") & (eval_table.bloc == "finale")][
    ["cible", "horizon", "modele", "n_previsions", "rmse_relative_modele", "rmse_relative_baseline", "biais_moyen_modele", "derive_systematique"]
])


,cible,horizon,modele,n_previsions,rmse_relative_modele,rmse_relative_baseline,biais_moyen_modele,derive_systematique
91,DETTE_PUBLIQUE,1,elasticnet,4,0.061275,0.032371,3.086693,True
93,DETTE_PUBLIQUE,1,panel_avec_effets_fixes,4,0.061934,0.032371,-0.428209,False
95,DETTE_PUBLIQUE,1,panel_leave_one_country_out,4,0.078198,0.032371,2.700399,False
97,DETTE_PUBLIQUE,2,elasticnet,4,0.143493,0.071290,2.919413,False
99,DETTE_PUBLIQUE,2,panel_avec_effets_fixes,4,0.100321,0.071290,-3.896014,True
101,DETTE_PUBLIQUE,2,panel_leave_one_country_out,4,0.099842,0.071290,0.955497,False
103,DETTE_PUBLIQUE,3,elasticnet,4,0.316729,0.106421,-2.848322,False
105,DETTE_PUBLIQUE,3,panel_avec_effets_fixes,4,0.157123,0.106421,-7.288553,True
107,DETTE_PUBLIQUE,3,panel_leave_one_country_out,4,0.143446,0.106421,-0.525272,False
109,DETTE_PUBLIQUE,4,elasticnet,4,0.241929,0.123381,-10.835370,True


**DETTE_PUBLIQUE** illustre le cas le plus difficile du portefeuille :
biais importants aux horizons longs, aucun modele ne bat clairement la
marche aleatoire sur la majorite des horizons -- cohérent avec une
dynamique de dette tres persistante et des chocs post-COVID inhabituels sur
la periode d'evaluation.

## 6. Objets sauvegardes pour l'etape 3

In [9]:
models_dir = Path(cfg2["paths"]["outputs_models_dir"])
pkl_files = sorted(models_dir.glob("elasticnet_*.pkl"))
print(f"{len(pkl_files)} objets de modele sauvegardes (7 cibles x 5 horizons x 2 blocs)")
with open(models_dir / "e2_model_registry.json", encoding="utf-8") as f:
    registry = json.load(f)
print("Exemple d'entree du registre (CROISSANCE_PIB, h=1) :")
registry["CROISSANCE_PIB_h1"]


70 objets de modele sauvegardes (7 cibles x 5 horizons x 2 blocs)
Exemple d'entree du registre (CROISSANCE_PIB, h=1) :


{'frozen_features': ['CROISSANCE_PIB_diff1',
  'CROISSANCE_PIB_lag0',
  'CROISSANCE_PIB_lag1',
  'CROISSANCE_PIB_lag2',
  'CROISSANCE_PIB_lag3',
  'CROISSANCE_PIB_ma3',
  'CROISSANCE_PIB_ma5',
  'CROISSANCE_PIB_std3',
  'CROISSANCE_PIB_std5',
  'FBCF_diff1',
  'FBCF_lag0',
  'FBCF_lag1',
  'FBCF_lag2',
  'FBCF_lag3',
  'FBCF_std3',
  'FBCF_std5',
  'IDEE_diff1',
  'IDEE_lag0',
  'IDEE_lag1',
  'IDEE_lag2',
  'IDEE_lag3',
  'IDEE_ma3',
  'IDEE_ma5',
  'IDEE_std3',
  'IDEE_std5'],
 'alpha': 0.3,
 'l1_ratio': 0.3,
 'tuning_detail': {'alpha': 0.3, 'l1_ratio': 0.3, 'n_folds': 5},
 'objets_deployables': {'interne': {'origin': 2019,
   'target_year': 2020,
   'features': ['CROISSANCE_PIB_diff1',
    'CROISSANCE_PIB_lag0',
    'CROISSANCE_PIB_lag1',
    'CROISSANCE_PIB_lag2',
    'CROISSANCE_PIB_lag3',
    'CROISSANCE_PIB_ma3',
    'CROISSANCE_PIB_ma5',
    'CROISSANCE_PIB_std3',
    'CROISSANCE_PIB_std5',
    'FBCF_diff1',
    'FBCF_lag0',
    'FBCF_lag1',
    'FBCF_lag2',
    'FBCF_lag3',
  

## 7. Bilan de l'etape 2

Voir `docs/ETAPE2_PROTOCOLE_ET_CARACTERISTIQUES.md` et
`docs/ETAPE2_MODELES_ET_VALIDATION.md` pour la synthese complete redigee,
`docs/MATRICE_CONFORMITE.md` (section Etape 2) pour chaque decision
documentee, et `docs/SUIVI_MODELISATION.md` pour l'avancement et les
prochaines actions avant l'etape 3.